In [ ]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
# import threading
mt5.initialize()


In [ ]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

In [ ]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 3, 1, tzinfo=timezone)
    utc_to = datetime(x.year, x.month+1, x.day+1, tzinfo=timezone)
    # utc_to = datetime(x.year, x.month+1, 1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_H1, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    
#     rates_frame['mean'] = (rates_frame['high'] + rates_frame['low'])/2

    # EMA = rates_frame['close'].ewm(span=100, adjust=False).mean()
    # DEMA = 2*EMA - EMA.ewm(span=100, adjust=False).mean()
    # rates_frame['dma'] = DEMA
#     rates_frame['P34'] = rates_frame['mean'].rolling(window=34).mean()
#     rates_frame['P5'] = rates_frame['mean'].rolling(window=5).mean()
#     rates_frame = rates_frame.fillna(0)
#     rates_frame['AO'] = rates_frame['P5'] - rates_frame['P34']
    rates_frame = rates_frame.drop(['high', 'low'], axis=1)
    rates_frame['rsi'] = pta.rsi(a['close'], length = 14)
    
    return rates_frame

In [ ]:
a = get_values("EURGBP")


In [ ]:
a.iloc[0].open-a.iloc[0].close

In [ ]:
p = []
check = 1
t = []
c = []
o = []
for i in range(1,len(a)-1):
    if str(a.iloc[i].rsi) != "nan":
        if a.iloc[i].rsi < 30 and check == 1:
            buy_price = a.iloc[i].close
            sell_price = a.iloc[i+1].close
            p.append(price_action("EURUSD", 1.0, buy_price, sell_price, mt5.ORDER_TYPE_SELL))
            t.append(a.iloc[i].name)
            c.append(a.iloc[i].rsi)
            o.append(a.iloc[i].open-a.iloc[i].close)
            
            check = 0
        if check == 0:
            if a.iloc[i].rsi > 30:
                check = 1

In [ ]:
sum(p)

In [ ]:
for i in range(0, len(p)):
    print(p[i],"--", c[i], "---", t[i],"***", o[i])

In [ ]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 4, 1, tzinfo=timezone)
    utc_to = datetime(x.year, x.month+1, x.day+1, tzinfo=timezone)
    # utc_to = datetime(x.year, x.month+1, 1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_H1, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    
    rates_frame['mean'] = (rates_frame['high'] + rates_frame['low'])/2

    # EMA = rates_frame['close'].ewm(span=100, adjust=False).mean()
    # DEMA = 2*EMA - EMA.ewm(span=100, adjust=False).mean()
    # rates_frame['dma'] = DEMA
    rates_frame['P34'] = rates_frame['mean'].rolling(window=34).mean()
    rates_frame['P5'] = rates_frame['mean'].rolling(window=5).mean()
    rates_frame['sma'] = rates_frame['close'].rolling(window=100).mean()
    
#     rates_frame = rates_frame.fillna(0)
    rates_frame['AO'] = rates_frame['P5'] - rates_frame['P34']
    rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    rates_frame = rates_frame.drop(['high', 'low', 'mean', 'P34', 'P5'], axis=1)
    return rates_frame

In [ ]:
symbol = "EURGBP"
a = get_values(symbol)

In [ ]:
l = ['NR']
for i in range(1,len(a)):
    df = a.iloc[i].AO
    dfo = a.iloc[i-1].AO
    if str(df) == 'nan':
        l.append('NR')
    else:
        if df <= 0.0:
            if df > dfo:
                l.append('NG')
            elif df < dfo:
                l.append("NR")
            else:
                print(type(df))
                l.append('R')
        elif df > 0.0:
            if df > dfo:
                l.append('PG')
            elif df < dfo:
                l.append("PR")
            else:
                print(type(df))
                l.append('NR')
        else:
            l.append('NR')
a['signal'] = l

In [ ]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
m = 0
n = 0
p= []
checks = 0
counterr = 0
counterp = 0
counterrp = 0
profits = []
l = 0

for i in range(7, len(a)):
    df = a.iloc[i].signal
    dfo = a.iloc[i-1].signal
    if str(a.iloc[i].sma) != 'nan':        
        if a.iloc[i].open < a.iloc[i].sma and a.iloc[i].close < a.iloc[i].sma \
        and a.iloc[i-1].open > a.iloc[i].sma and a.iloc[i-1].close < a.iloc[i].sma and check == 0:
            counter = 0
            counterr = 0
            m = 1
            
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*20)
            print(a.iloc[i].name)
            print("SELL")
            print("*"*20)
            check = 1    

        elif a.iloc[i].open < a.iloc[i].sma and a.iloc[i].close < a.iloc[i].sma \
        and a.iloc[i-1].open < a.iloc[i].sma and a.iloc[i-1].close > a.iloc[i].sma and check == 0:
            counter = 0
            counterr = 0
            m = 1
            
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*100)
            print(a.iloc[i].name)
            print("SELL")
            print("*"*100)
            check = 1    
            
            
        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            print(pp,"---",a.iloc[i].close)
#             profit.append(pp)
#             check = 0
            
            if pp > 0.0:
                counterr = counterr + 1
                if counterr > 2:
                    profit.append(pp)
                    check = 0
                    print(f"checks--> {checks}")
            if pp > 0.0 and m == 1:
                profit.append(pp)
                m = 0
            elif a.iloc[i].close > a.iloc[i].sma:
                if m == 1:
                    print("+"*100)
                    profit.append(2*pp)
                    check = 0
                    print(f"checks--> {checks}")
                else:
                    profit.append(pp)
                    check = 0  
                    print(f"checks--> {checks}")

#             elif pp < 0.0:
#                 counter = counter + 1
#                 if counter > 3:
#                     if m == 1:
#                         print("+"*100)
#                         profit.append(2*pp)
#                         checks = 0
#                     else:
#                         profit.append(pp)
#                         checks = 0         

        if a.iloc[i].open > a.iloc[i].sma and a.iloc[i].close > a.iloc[i].sma \
        and a.iloc[i-1].open < a.iloc[i].sma and a.iloc[i-1].close > a.iloc[i].sma and checks == 0:
            counterp = 0
            counterrp = 0
            n = 1
            
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*20)
            print(a.iloc[i].name)
            print("BUY")
            print("*"*20)
            checks = 1   
            
        elif a.iloc[i].open > a.iloc[i].sma and a.iloc[i].close > a.iloc[i].sma \
         and a.iloc[i-1].open > a.iloc[i].sma and a.iloc[i-1].close < a.iloc[i].sma \
         and checks == 0:
            counterp = 0
            counterrp = 0
            n = 1
            
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*100)
            print(a.iloc[i].name)
            print("BUY")
            print("*"*100)
            checks = 1   

        elif checks == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---",a.iloc[i].close)
#             profit.append(pp)
#             check = 0
            
            if pp > 0.0:
                counterrp = counterrp + 1
                if counterrp > 2:
                    profit.append(pp)
                    checks = 0
                    print(f"check--> {check}")
            if pp > 0.0 and n == 1:
                profit.append(pp)
                n = 0
            elif a.iloc[i].close < a.iloc[i].sma:
                if n == 1:
                    print("+"*100)
                    profit.append(2*pp)
                    checks = 0
                    print(f"check--> {check}")
                else:
                    profit.append(pp)
                    checks = 0   
                    print(f"check--> {check}")

            
#             elif pp < 0.0:
#                 counterp = counterp + 1
#                 if counter > 3:
#                     if n == 1:
#                         print("+"*100)
#                         profit.append(2*pp)
#                         checks = 0
#                     else:
#                         profit.append(pp)
#                         checks = 0         

In [ ]:
sum(profit)

In [ ]:
for i in range(len(profit)):
    if profit[i]!=0:
        print(profit[i])

In [ ]:
len(profit)-14

In [ ]:
        if df == "PG" and "G" in dfo and a.iloc[i].rsi > 66.0 and a.iloc[i-1].rsi < 66.0 and check ==0:
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)
            check = 1    

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action("EURGBP", 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp)
#             if pp > 0.30:
#                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
            if 'R' in df:
                print("R")
                check = 0
                index.append(a.iloc[i].name)
                profit.append(pp)
                print(a.iloc[i].name)
            elif pp < -2.0:
                print("pp")
                check = 0
                index.append(a.iloc[i].name)
                profit.append(pp)
                print(a.iloc[i].name)
#             elif a.iloc[i].rsi > 80.0:
#                 print("rsi")
#                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
#                 print(a.iloc[i].name)

            else:
                index.append(0)
                B.append('nan')
                profit.append(0)
                indexB.append(0)

In [ ]:
#             profit.append(pp)
#             print(a.iloc[i].name)
            
#             check = 0
#             if pp > 0.30:
# #                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
#             if 'R' in df:
#                 print("R")
#                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
#                 print(a.iloc[i].name)
#             if pp < -2.0:
#                 print("pp")
#                 check = 0
#                 index.append(a.iloc[i].name)
#                 profit.append(pp)
#                 print(a.iloc[i].name)

In [ ]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []

for i in range(7, len(a)):
    df = a.iloc[i].signal
    dfo = a.iloc[i-1].signal
    if str(a.iloc[i].AO) != 'nan':
        aa = a.iloc[i].rsi
        b = a.iloc[i-1].rsi
        c = aa-b
        
        height = abs(a.iloc[i].open - a.iloc[i].close)
#         if "G" in df and a.iloc[i].rsi > 52.0 and a.iloc[i-1].rsi < a.iloc[i].rsi and a.iloc[i-1].rsi < 50.0 \
#         and c > 2.0  and height > 0.00070  and check ==0:
#         if a.iloc[i].rsi > 47.0 and a.iloc[i].rsi < 50.0 and a.iloc[i-1].rsi < a.iloc[i].rsi and height > 0.00030:
        if df == "NG" and a.iloc[i].rsi > 51.0 and a.iloc[i-1].rsi < 50.0:
            B.append("buy")
            buy_price = a.iloc[i].close
            indexB.append(a.iloc[i].name)
            print("#"*20)
            print(a.iloc[i].name)
#             print(a.iloc[i].rsi)
            print("*"*20)
            check = 1    

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action("EURGBP", 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---",a.iloc[i].rsi)
            if a.iloc[i].rsi < 50.0:
                print("rsi")
                check = 0
                index.append(a.iloc[i].name)
                profit.append(pp)
                print(a.iloc[i].name)
            elif pp> 2.0:
                print("pp")
                check = 0
                index.append(a.iloc[i].name)
                profit.append(pp)
                print(a.iloc[i].name)
                
#         if df == "PR" and a.iloc[i].rsi < 49.0 and a.iloc[i-1].rsi >= 50.0:
#             B.append("buy")
#             buy_price = a.iloc[i].close
#             indexB.append(a.iloc[i].name)
#             print("#"*20)
#             print(a.iloc[i].name)
#             print(a.iloc[i].rsi)
#             print("*"*20)
#             checks = 1    

#         elif checks == 1:
#             sell_price = a.iloc[i].close
#             pp = price_action("EURGBP", 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
#             print(pp,"---",a.iloc[i].rsi)
#             if a.iloc[i].rsi >= 50.0:
#                 print("rsi")
#                 checks = 0
#                 index.append(a.iloc[i].name)
#                 profits.append(pp)
#                 print(a.iloc[i].name)

In [ ]:
q = 1
w = 2
e = 3
r = 4
if q==1 and w==2 or e==4 and r==4:
    print("df")